# 04 — GenAI: Follow-up Note Analysis and Recommendations

Run in Google Colab after Part 1 and Part 3. A free Gemini API key is required. Enter it only at runtime; never save it in the notebook or GitHub. The code preserves raw model replies and does not silently repair invalid JSON.

In [ ]:
!pip -q install -U google-genai pydantic

import json, time
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

# Use a currently available Flash model. If the class model was retired, select
# the current Flash model name shown in Google AI Studio.
MODEL_NAME = 'gemini-3.8-flash'
API_KEY = input('Paste your Google AI Studio API key (do not save this cell/output): ').strip()
client = genai.Client(api_key=API_KEY)

ALLOWED_CAUSES = [
    'forgot_or_no_reminder', 'distance_or_transport', 'cost_or_fee',
    'illness_or_emergency', 'work_or_schedule', 'other'
]
SCHEMA = {
    'type': 'object',
    'properties': {
        'root_cause': {'type': 'string'},
        'confidence': {'type': 'number'},
        'summary': {'type': 'string'},
        'recommended_action': {'type': 'string'}
    },
    'required': ['root_cause', 'confidence', 'summary', 'recommended_action']
}
SYSTEM_TEXT = (
    'You are a careful healthcare operations analytics assistant. '
    'Classify only what the note supports. Do not infer sensitive medical details. '
    'Use root_cause other when the note is ambiguous or insufficient.'
)

def build_prompt(note_text, version='CRAFT'):
    if version == 'V1':
        return 'Classify this note: ' + note_text
    return f'''Context: A patient missed a booked outpatient clinic appointment.
Role: You are a careful healthcare analytics assistant supporting the front desk.
Action: Identify the most likely reason stated in the note and suggest one practical follow-up.
Allowed root_cause values: {', '.join(ALLOWED_CAUSES)}.
Use other rather than guessing when evidence is ambiguous or insufficient.
Tone: Professional, empathetic, concise.
Format: Return JSON with exactly root_cause, confidence (0 to 1), summary (maximum 25 words), recommended_action.
Patient note: {note_text}'''

def call_gemini(prompt, max_attempts=3):
    # Gemini 3.8 Flash does not accept the older MINIMAL thinking-level setting.
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_TEXT,
        temperature=0,
        max_output_tokens=1200,
        response_mime_type='application/json',
        response_schema=SCHEMA
    )
    for attempt in range(max_attempts):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME, contents=prompt, config=config
            )
            return response.text
        except Exception as exc:
            message = str(exc).lower()
            # Retry only transient overload/service errors; do not hide auth/schema errors.
            transient = any(token in message for token in ['503', 'unavailable', 'overloaded', '429', 'resource_exhausted'])
            if not transient or attempt == max_attempts - 1:
                raise
            wait_seconds = 10 * (2 ** attempt)
            print(f'Gemini temporarily unavailable; retrying in {wait_seconds}s...')
            time.sleep(wait_seconds)
    raise RuntimeError('Gemini request failed after retries')

def analyse_note(note_text, version='CRAFT'):
    raw = call_gemini(build_prompt(note_text, version=version))
    return json.loads(raw), raw

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5, max_length=300)
    recommended_action: str = Field(min_length=5)

def validate_insight(data):
    insight = FollowupInsight(**data)
    if insight.root_cause not in ALLOWED_CAUSES:
        raise ValueError('root_cause is not in the allowed list')
    if len(insight.summary.split()) > 25:
        raise ValueError('summary must contain at most 25 words')
    return insight

api_call_count = 0
def counted_analysis(note_text, version='CRAFT'):
    global api_call_count
    api_call_count += 1  # count the request even if parsing/validation later fails
    data, raw = analyse_note(note_text, version=version)
    return data, raw

# Task 4.1: one validated call for N001
notes_df = pd.read_csv('/content/followup_notes.csv')
n001 = notes_df.loc[notes_df['note_id'] == 'N001', 'note_text'].iloc[0]
try:
    data_n001, raw_n001 = counted_analysis(str(n001))
    print(validate_insight(data_n001).model_dump())
except Exception as exc:
    data_n001, raw_n001 = None, str(exc)
    print('Task 4.1 API call unavailable; continue to the batch cell and preserve the error:', exc)


## Task 4.2 — Process first 10 notes with validation

In [ ]:
results = []
first10 = notes_df.head(10)
for i, row in first10.iterrows():
    note_text = row['note_text']
    record = {
        'note_id': row['note_id'], 'appointment_id': row['appointment_id'],
        'root_cause': '', 'confidence': None, 'summary': '',
        'recommended_action': '', 'validation_status': '', 'raw_response': ''
    }
    if pd.isna(note_text) or str(note_text).strip() == '':
        record.update(root_cause='other', confidence=0.0, summary='No note recorded',
                      recommended_action='Re-call patient', validation_status='skipped_blank')
    else:
        try:
            data, raw = counted_analysis(str(note_text))
            record['raw_response'] = raw
            # Copy exactly the fields returned; missing keys remain empty.
            for key in ['root_cause','confidence','summary','recommended_action']:
                if key in data:
                    record[key] = data[key]
            try:
                insight = validate_insight(data)
                record.update(insight.model_dump())
                record['validation_status'] = 'valid'
            except (ValidationError, ValueError) as validation_error:
                record['validation_status'] = 'invalid'
                print('Validation failure for', row['note_id'], str(validation_error))
        except Exception as api_error:
            record['validation_status'] = 'invalid'
            record['raw_response'] = 'API/PARSING ERROR: ' + str(api_error)
            print('API/parsing failure for', row['note_id'], str(api_error))
    results.append(record)
    time.sleep(13)  # free-tier rate limit; blank notes do not trigger an API call

notes_analysed = pd.DataFrame(results, columns=[
    'note_id','appointment_id','root_cause','confidence','summary',
    'recommended_action','validation_status','raw_response'
])
notes_analysed.to_csv('/content/outputs/notes_analysed.csv', index=False)
print(notes_analysed[['note_id','root_cause','confidence','validation_status']])


## Task 4.3 — Deliberately invalid replies and prompt comparison

In [ ]:
# Pydantic error test 1: confidence outside the permitted range
bad_reply_1 = {'root_cause':'other','confidence':1.7,'summary':'Unclear reason from note',
               'recommended_action':'Call the patient again'}
try:
    validate_insight(bad_reply_1)
except (ValidationError, ValueError) as exc:
    print('Bad reply 1 caught:', exc)

# Pydantic error test 2: required recommended_action missing
bad_reply_2 = {'root_cause':'other','confidence':0.5,'summary':'Unclear reason from note'}
try:
    validate_insight(bad_reply_2)
except (ValidationError, ValueError) as exc:
    print('Bad reply 2 caught:', exc)

# Compare V1 and CRAFT/V2 on N007 (Hinglish) and N009 (ambiguous).
comparison_rows = []
for note_id in ['N007','N009']:
    text = notes_df.loc[notes_df.note_id == note_id, 'note_text'].iloc[0]
    for version in ['V1','V2']:
        try:
            data, raw = counted_analysis(str(text), version=version)
            cause = data.get('root_cause', '')
            allowed = cause in ALLOWED_CAUSES
            comparison_rows.append({'note_id':note_id,'prompt':version,
                                    'root_cause':cause,'allowed':allowed,'raw_response':raw})
        except Exception as exc:
            comparison_rows.append({'note_id':note_id,'prompt':version,
                                    'root_cause':'','allowed':False,'raw_response':str(exc)})
        time.sleep(13)
prompt_comparison = pd.DataFrame(comparison_rows)
print(prompt_comparison[['note_id','prompt','root_cause','allowed']])
print('Total Gemini API calls across Tasks 4.1–4.3:', api_call_count)


## Task 4.4 — Merge ML risk with AI reasons and recommendations

In [ ]:
appointments = pd.read_csv('/content/outputs/appointments_clean.csv')
risk = pd.read_csv('/content/outputs/all_appointment_risk_scores.csv')
analysed = pd.read_csv('/content/outputs/notes_analysed.csv')

# Use all-appointment OOF/test scores so every note appointment can be matched when present.
merged = analysed.merge(appointments, on='appointment_id', how='left')
merged = merged.merge(risk[['appointment_id','risk_prob','flag']], on='appointment_id', how='left')
print('Merged note rows:', len(merged))
print('Root cause x reminder_sent:\n', pd.crosstab(merged['root_cause'], merged['reminder_sent']))
merged.to_csv('/content/outputs/final_recommendations.csv', index=False)


## Final interpretation

Use the root-cause distribution to tailor front-desk actions: reminder gaps call for reminder verification and confirmation; transport barriers call for rescheduling or transport guidance; cost concerns call for fee clarification or approved payment options; illness/emergency calls for empathetic rescheduling; work/schedule conflicts call for alternate slots. Prioritize appointments using the risk probability and the clinic's actual evening call capacity. Do not treat the 10 notes as a clinic-wide estimate.